<a href="https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dee0742/ML-FlyRank-Task/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



I will use Logistic Regression as the first learned model because the target is a yes/no observed label, `is_declining_label`.

Logistic Regression gives a simple and readable baseline for a classification problem. It also produces probabilities, which can be used to rank content items by their predicted likelihood of decline.

I will compare the learned model with the Week-4 rule baseline using the same split and evaluation metric. I will not use `trend_pct` or `trend_direction` because they are used to derive the label and would cause leakage.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [10]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load dataset
url = "https://raw.githubusercontent.com/dee0742/ML-FlyRank-Task/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)

# Create the target label
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

# Grouped train/test split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)

# Check that no client appears in both sets
shared_clients = set(train["client_id"]) & set(test["client_id"])

print("\nShared clients:", len(shared_clients))

Dataset shape: (30000, 44)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667

Train shape: (23837, 45)
Test shape: (6163, 45)

Shared clients: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

target = "is_declining_label"

# Columns that must never be model features
exclude = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

# Use numeric columns only for the first model
feature_columns = [
    col for col in df.columns
    if col not in exclude
    and pd.api.types.is_numeric_dtype(df[col])
]

print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 29
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


In [12]:
X_train = train[feature_columns]
y_train = train[target]

X_test = test[feature_columns]
y_test = test[target]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Probability of being a declining page
probability = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

Model trained successfully.


In [17]:
def precision_at_k(results, k):
    return results.head(k)["is_declining_label"].mean()


print("Model Precision@20:", precision_at_k(results, 20))
print("Model Precision@50:", precision_at_k(results, 50))
print("Model Precision@100:", precision_at_k(results, 100))

Model Precision@20: 1.0
Model Precision@50: 1.0
Model Precision@100: 1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [18]:
results = test[
    ["content_id", "client_id", target]
].copy()

results["predicted_probability"] = probability

results = results.sort_values(
    "predicted_probability",
    ascending=False
).reset_index(drop=True)

results.head(20)

,content_id,client_id,is_declining_label,predicted_probability
0,content_ca17a024f90c,client_4e07408562,1,1.0
1,content_007d1f134801,client_4e07408562,1,1.0
2,content_b138c0b35a91,client_4e07408562,1,1.0
3,content_60a90d0ba16a,client_f369cb89fc,1,1.0
4,content_c9147cda01f0,client_4e07408562,1,1.0
5,content_5fe46e04994d,client_4e07408562,1,1.0
6,content_d7175187ff12,client_4e07408562,1,1.0
7,content_05b68f8f80d4,client_4e07408562,1,1.0
8,content_8c19996aa890,client_4e07408562,1,1.0
9,content_e55da9707bf5,client_4e07408562,1,1.0


In [19]:
def precision_at_k(results, k):
    return results.head(k)[target].mean()

for k in [20, 50, 100]:
    print(
        f"Precision@{k}: "
        f"{precision_at_k(results, k):.4f}"
    )

Precision@20: 1.0000
Precision@50: 1.0000
Precision@100: 1.0000




The largest errors occur in observations where the predicted value differs substantially from the observed value.

The feature importance results show which available signals the Random Forest relies on most. These are model associations, not proof that the features cause the target.

The error analysis also shows that some observations remain difficult to predict. This suggests that improving the available signals or data quality may be more useful than simply increasing model complexity.

In [20]:
base_rate = y_test.mean()

print(f"Test-set base rate: {base_rate:.4f}")

Test-set base rate: 0.5110


In [24]:
comparison = pd.DataFrame({
    "Method": [
        "Logistic Regression"
    ],
    "Precision@20": [
        precision_at_k(results, 20)
    ],
    "Precision@50": [
        precision_at_k(results, 50)
    ],
    "Precision@100": [
        precision_at_k(results, 100)
    ],
    "Base rate": [
        base_rate
    ]
})

comparison

,Method,Precision@20,Precision@50,Precision@100,Base rate
0,Logistic Regression,1.0,1.0,1.0,0.510952


In [25]:
error_analysis = test[
    ["content_id", "client_id", target]
].copy()

error_analysis["predicted_probability"] = probability

error_analysis["predicted_label"] = (
    error_analysis["predicted_probability"] >= 0.5
).astype(int)

error_analysis["correct"] = (
    error_analysis["predicted_label"]
    == error_analysis[target]
)

error_analysis["error"] = (
    error_analysis[target]
    - error_analysis["predicted_probability"]
).abs()

error_analysis.sort_values(
    "error",
    ascending=False
).head(10)

,content_id,client_id,is_declining_label,predicted_probability,predicted_label,correct,error
24849,content_2f002563e9cd,client_e629fa6598,1,0.121598,0,False,0.878402
27487,content_31c66d071a62,client_8527a891e2,1,0.165393,0,False,0.834607
3488,content_a0777b0fd936,client_f369cb89fc,0,0.833677,1,False,0.833677
24218,content_a7c0affbca5a,client_e629fa6598,1,0.190730,0,False,0.809270
23644,content_dbde13469422,client_e629fa6598,1,0.199959,0,False,0.800041
7991,content_ce296f93e007,client_e629fa6598,1,0.201616,0,False,0.798384
17690,content_c268b1716236,client_e629fa6598,1,0.201917,0,False,0.798083
23511,content_4de8c62603bf,client_e629fa6598,1,0.202685,0,False,0.797315
7042,content_b3623d22db24,client_e629fa6598,1,0.202773,0,False,0.797227
29158,content_e18144cbd19d,client_4e07408562,1,0.202823,0,False,0.797177


In [26]:
top_50 = results.head(50)

false_positives = top_50[
    top_50[target] == 0
]

false_positives.head(3)

,content_id,client_id,is_declining_label,predicted_probability


In [27]:
classifier = model.named_steps["classifier"]

coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": classifier.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
).head(10)

,feature,coefficient,absolute_coefficient
15,impressions_last_30d,-34.951837,34.951837
18,impressions_prev_30d,29.131817,29.131817
5,impressions_90d,1.349930,1.349930
16,clicks_last_30d,-0.900605,0.900605
17,sessions_last_30d,-0.786321,0.786321
7,pageviews_90d,0.739676,0.739676
8,sessions_90d,0.731244,0.731244
9,users_90d,-0.706935,0.706935
19,clicks_prev_30d,0.700194,0.700194
13,days_with_impressions,0.545984,0.545984


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.